 # SignSage · FDMSE-ISL training v3: landmark augmentation + robust features

 Retrains the word model on **clean clips plus perturbed copies** that imitate live capture, using the same serving code that will run in the app.
 The live-path stress test (notebook 03) showed which conditions hurt `baseline_v1`: fewer frames per second, dropped frames, missed hands, a sign cut off or padded with idle frames, and a wrong aspect ratio. Each perturbation below is applied to some of the copies:

 | Perturbation (applied to the raw landmarks) | Share of copies |
 |---|---|
 | wrong aspect (x stretched, 0.8x to 1.25x) | 30% |
| camera roll (rotation, +-6 degrees) | 30% |
| detector noise (sigma 0.001 to 0.006) | 50% |
| hand dropout bursts (3% to 25% of frames, runs of about 2.5) | 50% |
| random frame drops (5% to 20%) | 30% |
| lower frame rate (22 to 47 frames) | 60% |
| window error (trim up to 20% OR pad up to 25% with idle frames) | 50% |

 Validation and test stay **clean and unchanged**, so the numbers are comparable with `baseline_v1` (test top-1 0.821). The real measure of this run is notebook 03 on it.

 **Before you start** - the clean features must be rebuilt with the robust hand handling (about 25 minutes, no GPU): open notebook 01 and run cells 1 → 2 → 3 → 4 → 5 → 6 → 8 → **12** (skip 9 and 10). That creates `features_v4r2_pose_face`.

 **Then here:** 1 Settings · 2 Setup · 3-6 write code · **7 Build augmented features** (30-60 min, resumable) · 8 Load · **9 Train** (resumable) · 10 Test once · 11 Final retrain on all signers · 12 Export.
 CPU runtime works (about 4 minutes per epoch, 30 epochs). A GPU runtime is faster, but only use it if you have quota to spare. Everything resumes after a disconnect: rerun cells 1 → 2 → 3 → 4 → 5 → 6 and then the cell you were in.

In [ ]:
# ============================ 1. SETTINGS (edit only here) ============================
RUN_NAME          = "aug_v1"           # new name = new run (own folder, own checkpoints)
OUT_ROOT          = None               # None -> the same folder the other notebooks used (Drive on Colab)
CLEAN_FEATURE_SET = "v4r2_pose_face"   # clean features with the ROBUST hand handling: build them first with notebook 01 cell 12
AUG_SET           = "aug_v1"           # folder for the augmented features built in cell 7
K_VARIANTS        = 4                  # augmented variants per clip
AUG_SEED          = 11

EPOCHS            = 30
EPOCH_SIZE        = 40000              # samples drawn per epoch from the (clean + augmented) pool
BATCH             = 128
LR                = 2e-3
WEIGHT_DECAY      = 1e-2
HIDDEN            = 128
LAYERS            = 2
DROPOUT           = 0.3
LABEL_SMOOTHING   = 0.1
PATIENCE          = 8
NOISE, MASK_P, SCALE_LO = 0.03, 0.05, 0.9   # in-training augmentation is lighter now: the landmark-level augmentation does the heavy work

DEVICE            = None               # None = GPU if the runtime has one, else CPU. A CPU runtime works (about 4 min per epoch).
THREADS           = None               # CPU threads; None = all cores
SEED              = 42
TARGET_PRECISION  = 0.90
RUN_FINAL_RETRAIN = True               # cell 11: retrain on ALL signers for deployment
FORCE_RETEST      = False              # keep False: the test split is meant to be used once

In [ ]:
# ============================ 2. SETUP ============================
import os, sys, json, importlib, time, hashlib
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_ROOT = OUT_ROOT or "/content/drive/MyDrive/SignSage_FDMSE_v2"
else:
    OUT_ROOT = OUT_ROOT or os.path.abspath("./signsage_fdmse_v2")
import numpy as np, pandas as pd, torch
sys.path.insert(0, os.getcwd())
DEV = DEVICE or ("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(THREADS or max(1, os.cpu_count() or 2))
META, CHUNKS = f"{OUT_ROOT}/meta", f"{OUT_ROOT}/chunks"
FEATS = f"{OUT_ROOT}/features_{CLEAN_FEATURE_SET}"
AUG_DIR = f"{OUT_ROOT}/features_{AUG_SET}"
RUN_DIR = f"{OUT_ROOT}/training/{RUN_NAME}"
os.makedirs(RUN_DIR, exist_ok=True)
for need in (f"{META}/manifest.csv", f"{META}/label_map.json", f"{META}/video_format.json", f"{FEATS}/feature_spec.json"):
    if not os.path.exists(need):
        raise SystemExit(f"Missing {need}. Run notebook 01 cell 12 first (it builds features_{CLEAN_FEATURE_SET}).")
FEATURE_SPEC = json.load(open(f"{FEATS}/feature_spec.json"))
if FEATURE_SPEC.get("hand_selection") != "robust":
    raise SystemExit(f"{FEATS} was built with the legacy hand handling. Run the updated notebook 01 cell 12 (ROBUST_HANDS = True).")
ASPECT = float(json.load(open(f"{META}/video_format.json"))["aspect"])
print(f"device {DEV} | cores {os.cpu_count()} | clean features: {FEATURE_SPEC['name']} (hands={FEATURE_SPEC['hand_selection']}, aspect={FEATURE_SPEC['aspect']})")
print(f"run folder: {RUN_DIR}")

In [ ]:
%%writefile sp_features.py
"""SignSage features: sign_features_v3 (VERBATIM from the original notebook /
backend/app/ml/preprocessing/sign_features_v3.py) + glue that rebuilds the
543-landmark layout from the raw landmarks saved by sp_core.py."""
import numpy as np

N_FRAMES = 60
FEAT_DIM = 122
MAX_GAP = 5
EPS = 1e-6

_LHAND = slice(501, 522)   # MediaPipe slot 0 (label ignored)
_RHAND = slice(522, 543)   # MediaPipe slot 1 (label ignored)
_FACE_NOSE = 1             # face mesh index; face array is landmarks[:,33:500,:2]
_FACE_CHIN = 152
_FACE_MOUTH = 13
_FACE_FOREHEAD = 10
_L_SHOULDER = 11
_R_SHOULDER = 12
_WRIST = 0
_INDEX_TIP = 8
_MIDDLE_TIP = 12
_FACE_POINTS = (_FACE_NOSE, _FACE_CHIN, _FACE_MOUTH, _FACE_FOREHEAD)


def _xy(landmarks, sl):
    return landmarks[:, sl, :2]


def _present(hand_xy):
    return np.any(hand_xy != 0.0, axis=(1, 2))


def _fill_gaps(hand_xy, max_gap=MAX_GAP):
    T = hand_xy.shape[0]
    filled = hand_xy.copy().astype(np.float64)
    present = _present(hand_xy)
    mask = present.copy().astype(np.float64)
    if T == 0:
        return filled, mask
    miss = ~present
    run_start = None
    for t in range(T + 1):
        if t < T and miss[t] and run_start is None:
            run_start = t
        if (t == T or not miss[t]) and run_start is not None:
            s, e = run_start, t - 1
            length = e - s + 1
            if length <= max_gap and s > 0 and e < T - 1:
                a = filled[s - 1]; b = filled[e + 1]
                frac = ((np.arange(s, e + 1, dtype=np.float64) - (s - 1))
                        / ((e + 1) - (s - 1)))
                for k in range(s, e + 1):
                    fk = frac[k - s]
                    filled[k] = a + fk * (b - a)
                mask[s:e + 1] = 1.0
            run_start = None
    return filled, mask


def _resample(feat, n_frames):
    T = feat.shape[0]
    if T == 0:
        return np.zeros((n_frames, FEAT_DIM), dtype=np.float32)
    if T == n_frames:
        return feat
    if T > n_frames:
        idx = np.linspace(0, T - 1, n_frames).round().astype(int)
        return feat[idx]
    pad = np.repeat(feat[-1:], n_frames - T, axis=0)
    return np.concatenate([feat, pad], axis=0)


def _shoulder_centre_width(pose):
    lsh = pose[:, _L_SHOULDER, :2]
    rsh = pose[:, _R_SHOULDER, :2]
    centre = (lsh + rsh) / 2.0
    width = np.linalg.norm(lsh - rsh, axis=1, keepdims=True) + EPS
    return centre, width


def _mirror_x(arr, mid_x):
    out = arr.copy()
    out[:, :, 0] = 2.0 * mid_x[:, None] - out[:, :, 0]
    return out


def _track_hands(lh_xy, rh_xy):
    T = lh_xy.shape[0]
    lh_present = _present(lh_xy)
    rh_present = _present(rh_xy)
    phys = []
    slots = [None, None]
    for t in range(T):
        detected = []
        if lh_present[t]:
            detected.append((lh_xy[t, _WRIST].copy(), lh_xy[t].copy()))
        if rh_present[t]:
            detected.append((rh_xy[t, _WRIST].copy(), rh_xy[t].copy()))
        phys_t = [None, None]
        if len(detected) == 0:
            pass
        elif len(detected) == 1:
            wrist, lm = detected[0]
            best_s, best_d = None, None
            for s in (0, 1):
                if slots[s] is not None:
                    d = float(np.linalg.norm(wrist - slots[s]["wrist"]))
                    if best_d is None or d < best_d:
                        best_s, best_d = s, d
            if best_s is None:
                best_s = 0
            phys_t[best_s] = lm
            slots[best_s] = {"wrist": wrist, "lm": lm}
        else:
            d0, d1 = detected
            if d0[0][0] > d1[0][0]:
                d0, d1 = d1, d0
            if slots[0] is not None and slots[1] is not None:
                c1 = (np.linalg.norm(d0[0] - slots[0]["wrist"])
                      + np.linalg.norm(d1[0] - slots[1]["wrist"]))
                c2 = (np.linalg.norm(d0[0] - slots[1]["wrist"])
                      + np.linalg.norm(d1[0] - slots[0]["wrist"]))
                if c1 <= c2:
                    phys_t[0], phys_t[1] = d0[1], d1[1]
                    slots[0] = {"wrist": d0[0], "lm": d0[1]}
                    slots[1] = {"wrist": d1[0], "lm": d1[1]}
                else:
                    phys_t[0], phys_t[1] = d1[1], d0[1]
                    slots[0] = {"wrist": d1[0], "lm": d1[1]}
                    slots[1] = {"wrist": d0[0], "lm": d0[1]}
            elif slots[0] is not None:
                if np.linalg.norm(d0[0] - slots[0]["wrist"]) <= np.linalg.norm(d1[0] - slots[0]["wrist"]):
                    phys_t[0], phys_t[1] = d0[1], d1[1]
                    slots[0] = {"wrist": d0[0], "lm": d0[1]}
                    slots[1] = {"wrist": d1[0], "lm": d1[1]}
                else:
                    phys_t[0], phys_t[1] = d1[1], d0[1]
                    slots[0] = {"wrist": d1[0], "lm": d1[1]}
                    slots[1] = {"wrist": d0[0], "lm": d0[1]}
            elif slots[1] is not None:
                if np.linalg.norm(d0[0] - slots[1]["wrist"]) <= np.linalg.norm(d1[0] - slots[1]["wrist"]):
                    phys_t[1], phys_t[0] = d0[1], d1[1]
                    slots[1] = {"wrist": d0[0], "lm": d0[1]}
                    slots[0] = {"wrist": d1[0], "lm": d1[1]}
                else:
                    phys_t[1], phys_t[0] = d1[1], d0[1]
                    slots[1] = {"wrist": d1[0], "lm": d1[1]}
                    slots[0] = {"wrist": d0[0], "lm": d0[1]}
            else:
                phys_t[0], phys_t[1] = d0[1], d1[1]
                slots[0] = {"wrist": d0[0], "lm": d0[1]}
                slots[1] = {"wrist": d1[0], "lm": d1[1]}
        phys.append(phys_t)
    wrists = np.zeros((T, 2, 2), dtype=np.float64)
    for t, phys_t in enumerate(phys):
        for s in (0, 1):
            if phys_t[s] is not None:
                wrists[t, s] = phys_t[s][_WRIST]
    return phys, wrists


def _active_secondary(phys, wrists):
    T = wrists.shape[0]
    travel = np.zeros(2)
    n_present = np.zeros(2, dtype=int)
    for s in (0, 1):
        present = np.any(wrists[:, s] != 0.0, axis=1)
        n_present[s] = int(present.sum())
        if n_present[s] > 1:
            travel[s] = float(np.sum(np.linalg.norm(np.diff(wrists[:, s], axis=0), axis=1)))
    if travel[0] > travel[1]:
        return 0
    if travel[1] > travel[0]:
        return 1
    return 1 if n_present[1] > n_present[0] else 0


def _should_mirror(phys, active, wrists, centre):
    T = wrists.shape[0]
    active_present = np.any(wrists[:, active] != 0.0, axis=1)
    sh_valid = np.any(centre != 0.0, axis=1)
    valid = active_present & sh_valid
    if valid.sum() < 2:
        return False
    return float(np.mean(wrists[valid, active, 0])) < float(np.mean(centre[valid, 0]))


def extract_features(landmarks, layout='signsage', n_frames=N_FRAMES):
    if layout != 'signsage':
        raise ValueError("sign_features_v3 supports layout='signsage' only (543-array)")
    if landmarks.ndim != 3 or landmarks.shape[2] < 2:
        raise ValueError("landmarks must be (T, N, >=3)")
    T = landmarks.shape[0]
    pose = landmarks[:, 0:33, :]
    lh = _xy(landmarks, _LHAND)
    rh = _xy(landmarks, _RHAND)
    face = landmarks[:, 33:500, :2]

    centre, width = _shoulder_centre_width(pose)
    phys, wrists = _track_hands(lh, rh)
    active = _active_secondary(phys, wrists)
    secondary = 1 - active

    mirror = _should_mirror(phys, active, wrists, centre)
    if mirror:
        mid_x = centre[:, 0]
        lh = _mirror_x(lh, mid_x)
        rh = _mirror_x(rh, mid_x)
        face = _mirror_x(face, mid_x)
        phys, wrists = _track_hands(lh, rh)
        active = _active_secondary(phys, wrists)
        secondary = 1 - active

    def hand_part(hfilled, hmask, hsize):
        wrist = hfilled[:, _WRIST, :]
        others = np.delete(hfilled, _WRIST, axis=1)
        wrs = (wrist - centre) / width
        orw = ((others - wrist[:, None, :]) / hsize.reshape(-1, 1, 1))
        wrs = wrs * hmask[:, None]
        orw = orw * hmask[:, None, None]
        return np.concatenate([wrs.reshape(T, -1), orw.reshape(T, -1)], axis=1)

    slot_lm = [np.zeros((T, 21, 2), dtype=np.float64),
               np.zeros((T, 21, 2), dtype=np.float64)]
    for t, phys_t in enumerate(phys):
        for s in (0, 1):
            if phys_t[s] is not None:
                slot_lm[s][t] = phys_t[s]

    l_filled, l_mask = _fill_gaps(slot_lm[0])
    r_filled, r_mask = _fill_gaps(slot_lm[1])

    def hand_size(hfilled):
        return np.linalg.norm(hfilled[:, _WRIST, :] - hfilled[:, _MIDDLE_TIP, :],
                              axis=1, keepdims=True) + EPS

    active_filled = l_filled if active == 0 else r_filled
    active_mask = l_mask if active == 0 else r_mask
    active_size = hand_size(active_filled)
    sec_filled = l_filled if secondary == 0 else r_filled
    sec_mask = l_mask if secondary == 0 else r_mask
    sec_size = hand_size(sec_filled)

    active_hand = hand_part(active_filled, active_mask, active_size)
    sec_hand = hand_part(sec_filled, sec_mask, sec_size)

    h2f = []
    for hp, hmask in ((active_filled[:, _WRIST, :], active_mask),
                      (active_filled[:, _INDEX_TIP, :], active_mask),
                      (sec_filled[:, _WRIST, :], sec_mask),
                      (sec_filled[:, _INDEX_TIP, :], sec_mask)):
        for fidx in _FACE_POINTS:
            h2f.append(((hp - face[:, fidx, :]) / width) * hmask[:, None])
    h2f = np.concatenate(h2f, axis=1)

    def wrist_vel(hfilled, hmask):
        wrist = hfilled[:, _WRIST, :]
        vel = np.zeros_like(wrist)
        if T > 1:
            vel[1:] = wrist[1:] - wrist[:-1]
        return vel * hmask[:, None]

    active_vel = wrist_vel(active_filled, active_mask)
    sec_vel = wrist_vel(sec_filled, sec_mask)

    feat = np.concatenate([active_hand, sec_hand,
                           active_mask.reshape(T, 1), sec_mask.reshape(T, 1),
                           h2f,
                           active_vel, sec_vel], axis=1)
    assert feat.shape[1] == FEAT_DIM, feat.shape[1]
    feat = _resample(feat, n_frames)
    return feat.astype(np.float32)


# ==========================================================================
# Robust variant of extract_features (sign_features_v4 revision 2; v3 above is unchanged).
# Fixes the two ways v3 mishandles MISSED hands: active-hand choice and mirroring. Identical when no hand is missed.
# ==========================================================================
def _active_secondary_robust(phys, wrists):
    """Which hand is the signing (active) hand. Same rule as v3 (larger wrist travel; ties by presence) but travel is
    measured ONLY between consecutive frames in which the hand was actually seen. v3 measures it on arrays where a
    missed frame is stored as zeros, so every miss adds two fake jumps of about |wrist| and a hand with a few
    dropouts looks like it moves a lot: the wrong hand is chosen and the whole sign may be mirrored."""
    travel = np.zeros(2)
    n_present = np.zeros(2, dtype=int)
    for s in (0, 1):
        present = np.any(wrists[:, s] != 0.0, axis=1)
        n_present[s] = int(present.sum())
        both = present[1:] & present[:-1]
        if both.any():
            step = np.linalg.norm(np.diff(wrists[:, s], axis=0), axis=1)
            travel[s] = float(np.sum(step[both]))
    if travel[0] > travel[1]:
        return 0
    if travel[1] > travel[0]:
        return 1
    return 1 if n_present[1] > n_present[0] else 0


def _mirror_x_present(arr, mid_x):
    """Mirror x about the shoulder centre for landmarks that were DETECTED only. v3 also mirrors the zeros that mean
    'not detected': x = 0 becomes 2 * mid_x, which is non-zero, so every missed hand frame turned into a phantom hand
    at the top of the image after mirroring."""
    out = arr.copy()
    seen = np.any(arr != 0.0, axis=2)                         # (T, K): landmark has any non-zero coordinate
    mirrored = 2.0 * mid_x[:, None] - arr[:, :, 0]
    out[:, :, 0] = np.where(seen, mirrored, 0.0)
    return out


def extract_features_robust(landmarks, layout='signsage', n_frames=N_FRAMES):
    if layout != 'signsage':
        raise ValueError("sign_features_v3 supports layout='signsage' only (543-array)")
    if landmarks.ndim != 3 or landmarks.shape[2] < 2:
        raise ValueError("landmarks must be (T, N, >=3)")
    T = landmarks.shape[0]
    pose = landmarks[:, 0:33, :]
    lh = _xy(landmarks, _LHAND)
    rh = _xy(landmarks, _RHAND)
    face = landmarks[:, 33:500, :2]

    centre, width = _shoulder_centre_width(pose)
    phys, wrists = _track_hands(lh, rh)
    active = _active_secondary_robust(phys, wrists)
    secondary = 1 - active

    mirror = _should_mirror(phys, active, wrists, centre)
    if mirror:
        mid_x = centre[:, 0]
        lh = _mirror_x_present(lh, mid_x)
        rh = _mirror_x_present(rh, mid_x)
        face = _mirror_x_present(face, mid_x)
        phys, wrists = _track_hands(lh, rh)
        active = _active_secondary_robust(phys, wrists)
        secondary = 1 - active

    def hand_part(hfilled, hmask, hsize):
        wrist = hfilled[:, _WRIST, :]
        others = np.delete(hfilled, _WRIST, axis=1)
        wrs = (wrist - centre) / width
        orw = ((others - wrist[:, None, :]) / hsize.reshape(-1, 1, 1))
        wrs = wrs * hmask[:, None]
        orw = orw * hmask[:, None, None]
        return np.concatenate([wrs.reshape(T, -1), orw.reshape(T, -1)], axis=1)

    slot_lm = [np.zeros((T, 21, 2), dtype=np.float64),
               np.zeros((T, 21, 2), dtype=np.float64)]
    for t, phys_t in enumerate(phys):
        for s in (0, 1):
            if phys_t[s] is not None:
                slot_lm[s][t] = phys_t[s]

    l_filled, l_mask = _fill_gaps(slot_lm[0])
    r_filled, r_mask = _fill_gaps(slot_lm[1])

    def hand_size(hfilled):
        return np.linalg.norm(hfilled[:, _WRIST, :] - hfilled[:, _MIDDLE_TIP, :],
                              axis=1, keepdims=True) + EPS

    active_filled = l_filled if active == 0 else r_filled
    active_mask = l_mask if active == 0 else r_mask
    active_size = hand_size(active_filled)
    sec_filled = l_filled if secondary == 0 else r_filled
    sec_mask = l_mask if secondary == 0 else r_mask
    sec_size = hand_size(sec_filled)

    active_hand = hand_part(active_filled, active_mask, active_size)
    sec_hand = hand_part(sec_filled, sec_mask, sec_size)

    h2f = []
    for hp, hmask in ((active_filled[:, _WRIST, :], active_mask),
                      (active_filled[:, _INDEX_TIP, :], active_mask),
                      (sec_filled[:, _WRIST, :], sec_mask),
                      (sec_filled[:, _INDEX_TIP, :], sec_mask)):
        for fidx in _FACE_POINTS:
            h2f.append(((hp - face[:, fidx, :]) / width) * hmask[:, None])
    h2f = np.concatenate(h2f, axis=1)

    def wrist_vel(hfilled, hmask):
        wrist = hfilled[:, _WRIST, :]
        vel = np.zeros_like(wrist)
        if T > 1:
            vel[1:] = wrist[1:] - wrist[:-1]
        return vel * hmask[:, None]

    active_vel = wrist_vel(active_filled, active_mask)
    sec_vel = wrist_vel(sec_filled, sec_mask)

    feat = np.concatenate([active_hand, sec_hand,
                           active_mask.reshape(T, 1), sec_mask.reshape(T, 1),
                           h2f,
                           active_vel, sec_vel], axis=1)
    assert feat.shape[1] == FEAT_DIM, feat.shape[1]
    feat = _resample(feat, n_frames)
    return feat.astype(np.float32)


# ==========================================================================
# Glue (new): raw saved landmarks -> (T, 543, 3) layout -> v3 features
# ==========================================================================
def _nearest_valid(ok):
    valid = np.flatnonzero(ok)
    if valid.size == 0:
        return None
    t = np.arange(ok.shape[0])
    return valid[np.abs(t[:, None] - valid[None, :]).argmin(axis=1)]


def interpolate_face(face, face_ok, n_frames, face_every_k):
    """Linear interpolation of the (sparse) face detections to every frame."""
    fidx = np.arange(0, n_frames, max(1, int(face_every_k)))
    good = np.flatnonzero(face_ok)
    if good.size == 0:
        return np.zeros((n_frames, 468, 3), np.float32)
    fx = fidx[good].astype(np.float64)
    F = face[good].astype(np.float32)
    t = np.arange(n_frames, dtype=np.float64)
    if good.size == 1:
        return np.repeat(F[:1], n_frames, axis=0)
    right = np.clip(np.searchsorted(fx, t, side="left"), 1, len(fx) - 1)
    left = right - 1
    w = np.clip((t - fx[left]) / (fx[right] - fx[left]), 0.0, 1.0).astype(np.float32)
    return (1.0 - w)[:, None, None] * F[left] + w[:, None, None] * F[right]


# --------------------------------------------------------------------------
# Face reference points from the POSE model (feature spec "sign_features_v4")
# MediaPipe's face model found a face in ~0% of frames of FDMSE-ISL, while pose found the person in
# 100%. sign_features_v3 only needs 4 face points (nose, chin, mouth, forehead); they are estimated
# from the pose model's head points: 0 nose, 2/5 eye centres, 9/10 mouth corners.
#   axis = eye_mid - mouth           (points up the face, follows head tilt)
#   chin = mouth - CHIN_K * axis,  forehead = eye_mid + FOREHEAD_K * axis   (rough anthropometric ratios)
# Serving must use exactly the same function on the pose part of the 543-landmark vector.
# --------------------------------------------------------------------------
FOREHEAD_K = 1.2
CHIN_K = 0.6


def pose_face_reference(pose_xy):
    """pose_xy: (T, >=11, >=2) -> {face_mesh_index: (T, 2)} for the four points v3 reads."""
    nose = pose_xy[:, 0, :2]
    eye = (pose_xy[:, 2, :2] + pose_xy[:, 5, :2]) / 2.0
    mouth = (pose_xy[:, 9, :2] + pose_xy[:, 10, :2]) / 2.0
    axis = eye - mouth
    return {_FACE_NOSE: nose, _FACE_CHIN: mouth - CHIN_K * axis,
            _FACE_MOUTH: mouth, _FACE_FOREHEAD: eye + FOREHEAD_K * axis}


def feature_spec(face_source="mesh", aspect=None, face_every_k=None, robust=False):
    """Provenance record stored next to the features and in the exported model."""
    return dict(name="sign_features_v4" if face_source == "pose" else "sign_features_v3",
                base="sign_features_v3 (unchanged code)", n_frames=N_FRAMES, feat_dim=FEAT_DIM,
                face_source=face_source, aspect=(round(float(aspect), 6) if aspect else None),
                hand_selection="robust" if robust else "legacy",
                forehead_k=FOREHEAD_K, chin_k=CHIN_K, face_every_k=face_every_k,
                note="aspect = video width/height; x coordinates are multiplied by it so x and y share one "
                     "unit. Serving must do the same with the live video's width/height.")


def to_landmarks_543(clip, face_every_k, face_source="mesh", aspect=None):
    """clip: dict with pose,pose_ok,hands,face,face_ok for ONE video.
    face_source 'mesh' = interpolate the saved face-mesh detections; 'pose' = derive the 4 reference
    points from the pose head points. aspect (= width/height) makes coordinates isotropic."""
    n = clip["pose"].shape[0]
    out = np.zeros((n, 543, 3), np.float32)
    pose = clip["pose"].copy()
    near = _nearest_valid(clip["pose_ok"])
    if near is not None:
        pose = pose[near]                      # fill missed pose frames from neighbours
    out[:, 0:33] = pose
    if face_source == "mesh":
        out[:, 33:501] = interpolate_face(clip["face"], clip["face_ok"], n, face_every_k)
    out[:, 501:522] = clip["hands"][:, 0]
    out[:, 522:543] = clip["hands"][:, 1]
    if aspect:
        out[:, :, 0] *= float(aspect)          # zeros (missing landmarks) stay zero
    if face_source == "pose":
        for idx, pts in pose_face_reference(out[:, 0:33]).items():
            out[:, 33 + idx, :2] = pts
    return out


def clip_quality(clip):
    hands_any = np.any(clip["hands"] != 0.0, axis=(2, 3)).any(axis=1)
    return dict(pose_ok_frac=float(clip["pose_ok"].mean()),
                hand_frac=float(hands_any.mean()),
                face_ok_frac=float(clip["face_ok"].mean()))


def is_usable(q, min_pose=0.5, min_hand=0.25):
    return q["pose_ok_frac"] >= min_pose and q["hand_frac"] >= min_hand


def features_for_clip(clip, face_every_k, face_source="mesh", aspect=None, robust=False):
    arr = to_landmarks_543(clip, face_every_k, face_source, aspect)
    return extract_features_robust(arr) if robust else extract_features(arr)


In [ ]:
%%writefile sp_train.py
"""SignSage FDMSE-ISL training (v2) - built for a small CPU (2 cores, 8 GB RAM).

* Reads only the saved feature chunks (feat_chunk_#####.npz) + the manifest: no video, no detectors.
* Resumable: a checkpoint is written atomically after every epoch.
* The test split is evaluated ONCE per run directory (a saved report guards against re-testing).
* Exports a self-describing bundle (weights, preprocessing, labels, metrics, model card).
"""
from __future__ import annotations

import hashlib
import json
import shutil
import tempfile
import zipfile
import math
import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

FEAT_RE = re.compile(r"^feat_chunk_(\d{5})\.npz$")
N_FRAMES, FEAT_DIM = 60, 122


# --------------------------------------------------------------------------
# data
# --------------------------------------------------------------------------
def _load_npz_retry(path, retries=6, log=print):
    """Copy to local disk first and retry: Google Drive's mount occasionally drops a read."""
    last = None
    for a in range(retries):
        tmp = None
        try:
            fd, tmp = tempfile.mkstemp(suffix=".npz")
            os.close(fd)
            shutil.copyfile(path, tmp)
            with np.load(tmp, allow_pickle=False) as z:
                return {k: z[k] for k in z.files}
        except (OSError, zipfile.BadZipFile, ValueError, EOFError, KeyError) as e:
            last = e
            log(f"  read retry {a+1}/{retries} for {Path(path).name}: {e.__class__.__name__}")
            time.sleep(min(2 ** (a + 1), 30))
        finally:
            if tmp and os.path.exists(tmp):
                os.unlink(tmp)
    raise SystemExit(f"Could not read {path} after {retries} tries ({last.__class__.__name__}). "
                     "Rerun the cell; if it persists remount Drive (restart the runtime and run cell 2).")


def load_dataset(feat_dir, manifest, log=print):
    """Return dict(X float16 (N,60,122), y int64, split str, signer str, stats)."""
    feat_dir = Path(feat_dir)
    files = sorted(p for p in feat_dir.iterdir() if FEAT_RE.match(p.name)) if feat_dir.is_dir() else []
    if not files:
        raise SystemExit(f"No feature files in {feat_dir}. Run the extraction notebook (cells 10 and 12) first.")
    key_to_row = {hashlib.sha1(p.encode()).hexdigest(): i for i, p in enumerate(manifest["path"])}
    cls = manifest["class_id"].to_numpy()
    spl = manifest["split"].to_numpy()
    sig = manifest["signer"].to_numpy()
    Xs, rows_ok, rows_seen, qs = [], [], [], []
    for p in files:
        z = _load_npz_retry(p, log=log)
        keys = z["keys"].astype(str)
        rows = np.array([key_to_row.get(k, -1) for k in keys], dtype=np.int64)
        known = rows >= 0
        rows_seen.append(rows[known])
        sel = known & z["usable"].astype(bool)
        Xs.append(z["X"][sel].astype(np.float16))
        qs.append(z["quality"][sel].astype(np.float32))
        rows_ok.append(rows[sel])
    X = np.concatenate(Xs)
    rows_ok = np.concatenate(rows_ok)
    rows_seen = np.concatenate(rows_seen)
    if X.shape[1:] != (N_FRAMES, FEAT_DIM):
        raise SystemExit(f"Unexpected feature shape {X.shape[1:]}; expected {(N_FRAMES, FEAT_DIM)}")
    out = dict(X=X, y=cls[rows_ok].astype(np.int64), split=spl[rows_ok], signer=sig[rows_ok], rows=rows_ok,
               quality=np.concatenate(qs))
    n_cls = int(manifest["class_id"].max()) + 1
    seen_split = pd.Series(spl[rows_seen]).value_counts().to_dict()
    ok_split = pd.Series(out["split"]).value_counts().to_dict()
    stats = dict(n_classes=n_cls, converted=int(len(rows_seen)), usable=int(len(rows_ok)),
                 per_split_converted={k: int(v) for k, v in seen_split.items()},
                 per_split_usable={k: int(v) for k, v in ok_split.items()},
                 total_in_manifest=int(len(manifest)), feature_files=len(files))
    for sp in ("train", "validation", "test"):
        m = out["split"] == sp
        stats[f"classes_in_{sp}"] = int(len(np.unique(out["y"][m]))) if m.any() else 0
    out["stats"] = stats
    log(f"feature files: {len(files)} | clips converted: {stats['converted']}/{stats['total_in_manifest']} "
        f"| usable: {stats['usable']}")
    for sp in ("train", "validation", "test"):
        c, u = seen_split.get(sp, 0), ok_split.get(sp, 0)
        log(f"  {sp:<10} converted {c:>6} | usable {u:>6} | unusable {c-u:>5} | classes present {stats[f'classes_in_{sp}']}/{n_cls}")
    return out


def quality_report(data, std=None, log=print):
    """How well MediaPipe saw each clip, and whether any feature is dead. Returns a list of warnings."""
    q = data["quality"]
    warnings = []
    log("detection quality over all usable clips (fraction of frames):")
    for j, name in enumerate(("pose detected", "hands visible", "face detected")):
        v = q[:, j]
        log(f"  {name:<14} mean {v.mean():.3f} | median {np.median(v):.3f} | 5th percentile {np.percentile(v, 5):.3f}"
            f" | clips below 0.5: {(v < 0.5).mean():.1%}")
    if np.median(q[:, 2]) < 0.8:
        warnings.append(
            "FACE was detected in fewer than 80% of frames (median). The hand-to-face features would be built "
            "from missing face points, and the live app (which sees real faces) would then give the model "
            "inputs it never saw in training. Do not trust the results until this is fixed - see the notes in "
            "the notebook.")
    if np.median(q[:, 1]) < 0.7:
        warnings.append("Hands are visible in fewer than 70% of frames (median): check the hand detection.")
    if std is not None:
        dead = np.flatnonzero(std <= 1.0001e-3)
        log(f"feature columns with no variation (dead): {len(dead)} of {FEAT_DIM}" + (f" -> {dead.tolist()}" if len(dead) else ""))
        if len(dead) > 10:
            warnings.append(f"{len(dead)} of {FEAT_DIM} features never change - part of the input is empty.")
    for w in warnings:
        log("WARNING: " + w)
    if not warnings:
        log("quality check passed (no warnings).")
    return warnings


def fit_standardizer(X_train):
    """Per-feature mean/std over every frame of the training clips (float32 accumulation)."""
    n = 0
    s = np.zeros(FEAT_DIM, np.float64)
    ss = np.zeros(FEAT_DIM, np.float64)
    for i in range(0, len(X_train), 2048):
        b = X_train[i:i + 2048].astype(np.float32).reshape(-1, FEAT_DIM)
        s += b.sum(0)
        ss += (b.astype(np.float64) ** 2).sum(0)
        n += len(b)
    mean = s / n
    std = np.sqrt(np.maximum(ss / n - mean ** 2, 1e-12))
    return mean.astype(np.float32), np.maximum(std, 1e-3).astype(np.float32)


# --------------------------------------------------------------------------
# model
# --------------------------------------------------------------------------
class SignGRU(nn.Module):
    """Linear projection -> 2-layer BiGRU -> attention + max pooling -> classifier."""

    def __init__(self, n_classes, in_dim=FEAT_DIM, proj=128, hidden=128, layers=2, dropout=0.3):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(in_dim, proj), nn.GELU(), nn.Dropout(dropout * 0.5))
        self.gru = nn.GRU(proj, hidden, num_layers=layers, batch_first=True, bidirectional=True,
                          dropout=dropout if layers > 1 else 0.0)
        self.attn = nn.Linear(2 * hidden, 1)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(4 * hidden, n_classes))

    def forward(self, x):
        h, _ = self.gru(self.proj(x))
        w = torch.softmax(self.attn(h).squeeze(-1), dim=1).unsqueeze(-1)
        return self.head(torch.cat([(h * w).sum(1), h.max(1).values], dim=1))


def build_model(arch):
    return SignGRU(arch["n_classes"], FEAT_DIM, arch["proj"], arch["hidden"], arch["layers"], arch["dropout"])


def n_params(model):
    return sum(p.numel() for p in model.parameters())


def augment(x, gen, noise=0.05, mask_p=0.10, scale_lo=0.8):
    """Time-crop-and-stretch (speed jitter), Gaussian noise and random frame dropout."""
    b, t, _ = x.shape
    s = float(torch.empty(1).uniform_(scale_lo, 1.0, generator=gen))
    length = max(8, int(round(t * s)))
    if length < t:
        start = int(torch.randint(0, t - length + 1, (1,), generator=gen))
        x = F.interpolate(x[:, start:start + length].transpose(1, 2), size=t, mode="linear",
                          align_corners=False).transpose(1, 2)
    if noise:
        x = x + noise * torch.randn(x.shape, generator=gen)
    if mask_p:
        x = x.masked_fill(torch.rand(b, t, 1, generator=gen) < mask_p, 0.0)
    return x


def lr_at(step, total, base, warmup):
    if step < warmup:
        return base * (step + 1) / warmup
    p = (step - warmup) / max(1, total - warmup)
    return base * (0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * min(1.0, p))))


# --------------------------------------------------------------------------
# evaluation helpers
# --------------------------------------------------------------------------
@torch.no_grad()
def predict_probs(model, X, mean, std, batch=512):
    model.eval()
    dev = next(model.parameters()).device
    m, s = torch.from_numpy(mean), torch.from_numpy(std)
    out = []
    for i in range(0, len(X), batch):
        xb = ((torch.from_numpy(X[i:i + batch]).float() - m) / s).to(dev)
        out.append(torch.softmax(model(xb), dim=1).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, 1), np.float32)


def metrics_from_probs(probs, y, n_classes):
    pred = probs.argmax(1)
    top1 = float((pred == y).mean())
    k = min(5, probs.shape[1])
    topk = np.argpartition(-probs, k - 1, axis=1)[:, :k]
    top5 = float((topk == y[:, None]).any(1).mean())
    nll = float(-np.log(np.maximum(probs[np.arange(len(y)), y], 1e-9)).mean())
    recalls = []
    for c in np.unique(y):
        recalls.append(float((pred[y == c] == c).mean()))
    return dict(top1=top1, top5=top5, loss=nll, macro_recall=float(np.mean(recalls)),
                n=int(len(y)), n_classes_present=int(len(np.unique(y))))


def per_class_report(probs, y, classes):
    pred = probs.argmax(1)
    rows = []
    for c in np.unique(y):
        m = y == c
        wrong = pred[m][pred[m] != c]
        conf = int(np.bincount(wrong).argmax()) if len(wrong) else -1
        rows.append(dict(class_id=int(c), word=classes[c], n=int(m.sum()), recall=float((pred[m] == c).mean()),
                         most_confused_with=classes[conf] if conf >= 0 else ""))
    return pd.DataFrame(rows).sort_values("recall")


def calibrate_threshold(probs, y, target_precision=0.90, min_accepted=50, floor=0.02):
    """Smallest confidence threshold (not below `floor`) whose accepted predictions reach
    `target_precision` on the given (validation) data. A correct model can still be modest in
    confidence, so no large fixed floor is imposed."""
    conf = probs.max(1)
    pred = probs.argmax(1)
    ok = pred == y
    table = []
    for t in np.round(np.arange(0.05, 0.99, 0.05), 2):
        acc = conf >= t
        table.append(dict(threshold=float(t), coverage=float(acc.mean()),
                          precision=float(ok[acc].mean()) if acc.any() else float("nan")))
    chosen = None
    for t in np.round(np.arange(floor, 1.0, 0.01), 2):
        acc = conf >= t
        if acc.sum() >= min_accepted and ok[acc].mean() >= target_precision:
            chosen = float(t)
            break
    return chosen, pd.DataFrame(table)


def rejection_checks(model, mean, std, X_ref, threshold, seed=0):
    """Inputs that are NOT signs must stay below the confidence threshold."""
    g = np.random.RandomState(seed)
    m, s = torch.from_numpy(mean), torch.from_numpy(std)
    ref = X_ref[g.randint(len(X_ref))].astype(np.float32)
    cases = {
        "all zeros": np.zeros((1, N_FRAMES, FEAT_DIM), np.float32),
        "random noise": g.randn(1, N_FRAMES, FEAT_DIM).astype(np.float32) * std + mean,
        "frozen pose (one frame repeated)": np.repeat(ref[:1][None], N_FRAMES, axis=1),
        "5 frames only (rest zeros)": np.concatenate([ref[:5], np.zeros((N_FRAMES - 5, FEAT_DIM), np.float32)])[None],
    }
    model.eval()
    dev = next(model.parameters()).device
    res = {}
    with torch.no_grad():
        for name, arr in cases.items():
            p = torch.softmax(model(((torch.from_numpy(arr) - m) / s).to(dev)), dim=1)[0].cpu()
            res[name] = dict(max_confidence=float(p.max()),
                             rejected=bool(threshold is not None and float(p.max()) < threshold))
    return res


# --------------------------------------------------------------------------
# training (resumable)
# --------------------------------------------------------------------------
def _atomic_save(obj, path):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def fingerprint(hp, n_train, n_classes):
    hp = {k: v for k, v in hp.items() if k != "device"}              # resuming on another device is fine
    core = dict(hp, n_train=int(n_train), n_classes=int(n_classes), feat=(N_FRAMES, FEAT_DIM))
    return hashlib.sha1(json.dumps(core, sort_keys=True).encode()).hexdigest()[:12]


def train(arch, hp, data, run_dir, mean, std, X_val=None, y_val=None, fixed_epochs=None, log=print,
          log_file=None):
    """Train (or resume) in run_dir. data = (X_train float16, y_train int64).
    With X_val: early stopping on validation top-1. Without: runs fixed_epochs, keeps the last weights."""
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    X, y = data
    fp = fingerprint(dict(hp, **{k: arch[k] for k in ("proj", "hidden", "layers", "dropout")}), len(X), arch["n_classes"])
    epochs = fixed_epochs or hp["epochs"]
    if log_file:
        _p = log

        def log(msg, _q=_p):
            _q(msg)
            try:
                with open(log_file, "a", encoding="utf-8") as fh:
                    fh.write(time.strftime("%Y-%m-%d %H:%M:%S ") + str(msg) + "\n")
            except OSError:
                pass

    torch.manual_seed(hp["seed"])
    dev = torch.device(hp.get("device") or "cpu")
    model = build_model(arch)
    opt = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])
    gen = torch.Generator().manual_seed(hp["seed"])
    state = dict(epoch=0, step=0, best=-1.0, best_epoch=0, bad=0, history=[], fp=fp)
    last = run_dir / "last.pt"
    if last.exists():
        ck = torch.load(last, map_location="cpu", weights_only=False)
        if ck["state"]["fp"] != fp:
            raise SystemExit(
                f"{run_dir} holds a run with different settings or data (fingerprint {ck['state']['fp']} vs {fp}). "
                "Use a new RUN_NAME, or restore the previous settings.")
        model.load_state_dict(ck["model"])
        model.to(dev)
        opt = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])
        opt.load_state_dict(ck["opt"])
        gen.set_state(ck["gen"])
        state = ck["state"]
        log(f"resuming {run_dir.name} at epoch {state['epoch']}/{epochs} (best so far {state['best']:.4f})")
        if state["epoch"] >= epochs or (X_val is not None and state["bad"] >= hp["patience"]):
            log("this run is already finished.")
    model.to(dev)
    epoch_size = int(hp["epoch_size"]) if hp.get("epoch_size") else None
    log(f"model parameters: {n_params(model):,} | train clips: {len(X):,}"
        + (f" (sampling {epoch_size:,} per epoch)" if epoch_size else "")
        + f" | batch {hp['batch']} | epochs <= {epochs} | device {dev}")

    m_t, s_t = torch.from_numpy(mean), torch.from_numpy(std)
    y_t = torch.from_numpy(y)
    n_pool = len(X)
    n = min(n_pool, epoch_size) if epoch_size else n_pool            # samples seen per epoch
    steps_per_epoch = math.ceil(n / hp["batch"])
    total_steps = steps_per_epoch * epochs
    warm = min(300, max(10, total_steps // 20))
    t_start = time.time()
    done_now = 0
    while state["epoch"] < epochs and not (X_val is not None and state["bad"] >= hp["patience"]):
        ep = state["epoch"] + 1
        t0 = time.time()
        model.train()
        perm = torch.randperm(n_pool, generator=gen)[:n]
        tot_loss = tot_ok = 0.0
        for i in range(0, n, hp["batch"]):
            idx = perm[i:i + hp["batch"]].numpy()
            idx.sort()                                         # sorted reads from the float16 array
            xb = (torch.from_numpy(X[idx]).float() - m_t) / s_t
            xb = augment(xb, gen, hp["noise"], hp["mask_p"], hp["scale_lo"]).to(dev)
            yb = y_t[idx].to(dev)
            for g_ in opt.param_groups:
                g_["lr"] = lr_at(state["step"], total_steps, hp["lr"], warm)
            logits = model(xb)
            loss = F.cross_entropy(logits, yb, label_smoothing=hp["label_smoothing"])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            state["step"] += 1
            tot_loss += float(loss.detach()) * len(idx)
            tot_ok += float((logits.argmax(1) == yb).sum())
        state["epoch"] = ep
        rec = dict(epoch=ep, train_loss=tot_loss / n, train_acc=tot_ok / n, lr=opt.param_groups[0]["lr"])
        msg = f"epoch {ep:>3}/{epochs} | train loss {rec['train_loss']:.3f} acc {rec['train_acc']:.3f}"
        if X_val is not None:
            pv = predict_probs(model, X_val, mean, std)
            mv = metrics_from_probs(pv, y_val, arch["n_classes"])
            rec.update(val_top1=mv["top1"], val_top5=mv["top5"], val_loss=mv["loss"])
            msg += f" | val top1 {mv['top1']:.3f} top5 {mv['top5']:.3f} loss {mv['loss']:.3f}"
            if mv["top1"] > state["best"] + 1e-6:
                state.update(best=mv["top1"], best_epoch=ep, bad=0)
                _atomic_save(dict(model=model.state_dict(), epoch=ep, val=mv), run_dir / "best.pt")
                msg += "  *best*"
            else:
                state["bad"] += 1
        rec["seconds"] = time.time() - t0
        state["history"].append(rec)
        done_now += 1
        eta = (time.time() - t_start) / done_now * (epochs - ep) / 60
        log(msg + f" | {rec['seconds']:.0f}s | ETA <= {eta:.0f} min")
        with open(run_dir / "metrics.jsonl", "a", encoding="utf-8") as fh:
            fh.write(json.dumps(rec) + "\n")
        _atomic_save(dict(model=model.state_dict(), opt=opt.state_dict(), gen=gen.get_state(), state=state), last)
    if X_val is None:
        _atomic_save(dict(model=model.state_dict(), epoch=state["epoch"]), run_dir / "final.pt")
    log(f"finished: {state['epoch']} epoch(s)" + (f", best validation top-1 {state['best']:.4f} at epoch {state['best_epoch']}"
                                                   if X_val is not None else ""))
    return model, state


# --------------------------------------------------------------------------
# export
# --------------------------------------------------------------------------
def export_bundle(out_dir, model, arch, mean, std, classes, extraction_cfg, metrics, threshold, notes, feature_spec=None):
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, out / "model.pt")
    (out / "model_config.json").write_text(json.dumps(dict(architecture="SignGRU", **arch), indent=2))
    (out / "preprocessing.json").write_text(json.dumps(dict(
        feature_spec=(feature_spec or {}).get("name", "sign_features_v3"), feature_spec_detail=feature_spec,
        n_frames=N_FRAMES, feat_dim=FEAT_DIM,
        standardize_mean=mean.tolist(), standardize_std=std.tolist(),
        extraction=extraction_cfg,
        note="Serving must sample frames and extract landmarks exactly like `extraction`, then apply "
             "sign_features_v3 and (x - mean) / std."), indent=1))
    (out / "labels.json").write_text(json.dumps(dict(classes=list(classes)), ensure_ascii=False, indent=1), encoding="utf-8")
    (out / "metrics.json").write_text(json.dumps(dict(metrics, confidence_threshold=threshold), indent=2))
    card = [
        "# SignSage FDMSE-ISL isolated-word model", "",
        f"* Architecture: SignGRU, {n_params(model):,} parameters, {arch['n_classes']} classes",
        f"* Input: {(feature_spec or {}).get('name', 'sign_features_v3')}, {N_FRAMES} x {FEAT_DIM}", f"* Confidence threshold: {threshold}", "",
        "## Data", "Trained on FDMSE-ISL (Patra et al., Pattern Analysis and Applications 28(3):148, 2025), "
        "used with the authors' permission. Weights follow the dataset's terms and do not inherit the code's MIT licence.", "",
        "## Results", "```json", json.dumps(metrics, indent=2), "```", "", "## Notes", notes, "",
        "## Limits", "Studio-recorded signers; webcam performance is unmeasured. Not validated for real-world use. "
        "Not promoted until the promotion gates in docs/MODELS_AND_DATA.md pass."]
    (out / "MODEL_CARD.md").write_text("\n".join(card), encoding="utf-8")
    return out


def load_bundle(bundle_dir):
    b = Path(bundle_dir)
    arch = json.loads((b / "model_config.json").read_text())
    pre = json.loads((b / "preprocessing.json").read_text())
    labels = json.loads((b / "labels.json").read_text(encoding="utf-8"))["classes"]
    met = json.loads((b / "metrics.json").read_text())
    model = build_model(arch)
    model.load_state_dict(torch.load(b / "model.pt", map_location="cpu"))
    model.eval()
    return dict(model=model, mean=np.array(pre["standardize_mean"], np.float32),
                std=np.array(pre["standardize_std"], np.float32), classes=labels,
                threshold=met.get("confidence_threshold"))


def predict_topk(bundle, feats, k=3):
    """feats: (60,122) sign_features_v3 array -> list of (word, probability), or [] if below the threshold."""
    p = predict_probs(bundle["model"], feats[None].astype(np.float32), bundle["mean"], bundle["std"])[0]
    top = np.argsort(-p)[:k]
    thr = bundle["threshold"]
    if thr is not None and p[top[0]] < thr:
        return []
    return [(bundle["classes"][i], float(p[i])) for i in top]


In [ ]:
%%writefile serving_v4.py
# COPY of backend/app/ml/preprocessing/sign_features_v4.py (only the import path differs). Keep in sync.
"""sign_features_v4 - serving-side recipe for the FDMSE-ISL word models.

The model was trained on features built by this exact recipe (see docs/MODELS_AND_DATA.md):

  1. Resample the captured landmark sequence to EXACTLY `n_frames` (48) frames, uniformly over the
     capture. sign_features_v3 pads a shorter sequence by repeating the LAST frame, and its wrist
     velocities are per frame step, so feeding it ~25 live frames instead of the 48 it was trained
     on would change both the padding and the velocity scale.
  2. Fill frames where the pose is missing from the nearest frame that has it.
  3. Multiply x by the video aspect ratio (width / height) so x and y share one unit.
  4. Derive the four face reference points (nose, chin, mouth, forehead) from the POSE head points.
     The face mesh is deliberately ignored: MediaPipe never found a face in the training videos.
  5. Run sign_features_v3's feature code on the result. The default (robust=True) differs from v3 in ONE place:
     which hand is "active" is decided from frames where the hand was seen (v3 counts a missed frame as a jump to
     the image corner, so a hand with a few dropouts looks like the signing hand). Without missed hands the two
     give identical features.

Any change here must be mirrored in the training code and the golden-vector test.
"""
from __future__ import annotations

import numpy as np

from sp_features import (
    EPS,
    FEAT_DIM,
    N_FRAMES,
    _FACE_CHIN,
    _FACE_FOREHEAD,
    _FACE_MOUTH,
    _FACE_NOSE,
    _FACE_POINTS,
    _INDEX_TIP,
    _LHAND,
    _MIDDLE_TIP,
    _RHAND,
    _WRIST,
    _fill_gaps,
    _mirror_x,
    _resample,
    _shoulder_centre_width,
    _should_mirror,
    _track_hands,
    _xy,
    extract_features as _extract_v3,
)

EXTRACT_FRAMES = 48          # frames the training pipeline sampled per clip
FOREHEAD_K = 1.2
CHIN_K = 0.6
_MAX_LINEAR_JUMP = 0.3       # do not interpolate a hand that "jumped" > 30% of the frame between samples

_POSE = slice(0, 33)
_FACE = slice(33, 501)
_LHAND = slice(501, 522)
_RHAND = slice(522, 543)


def _block_valid(block: np.ndarray) -> np.ndarray:
    """(T, K, 3) -> (T,) True where the block has any non-zero coordinate (zeros = not detected)."""
    return np.any(block != 0.0, axis=(1, 2))


def resample_landmarks(arr: np.ndarray, n: int = EXTRACT_FRAMES) -> np.ndarray:
    """Uniformly resample (T, 543, 3) to (n, 543, 3).

    Each body block (pose, left hand, right hand) is interpolated linearly only when it is present in
    both neighbouring frames and did not jump; otherwise the nearest frame is used, and a block that is
    missing in the nearest frame stays missing (never invented). T == n returns an exact copy."""
    t_in = arr.shape[0]
    if t_in == n:
        return np.array(arr, dtype=np.float32, copy=True)
    out = np.zeros((n, 543, 3), np.float32)
    t = np.linspace(0.0, t_in - 1, n)
    i0 = np.floor(t).astype(int)
    i1 = np.minimum(i0 + 1, t_in - 1)
    w = (t - i0).astype(np.float32)
    near_idx = np.where(w < 0.5, i0, i1)
    for sl in (_POSE, _LHAND, _RHAND):
        blk = arr[:, sl]
        valid = _block_valid(blk)
        a, b = blk[i0], blk[i1]
        jump = np.abs(a[..., :2] - b[..., :2]).mean(axis=(1, 2))
        linear = valid[i0] & valid[i1] & (jump <= _MAX_LINEAR_JUMP)
        blended = (1.0 - w)[:, None, None] * a + w[:, None, None] * b
        nearest = np.where(valid[near_idx][:, None, None], blk[near_idx], 0.0)
        out[:, sl] = np.where(linear[:, None, None], blended, nearest)
    return out


def _fill_missing_pose(pose: np.ndarray) -> np.ndarray:
    ok = _block_valid(pose)
    if ok.all() or not ok.any():
        return pose
    idx = np.flatnonzero(ok)
    t = np.arange(len(pose))
    return pose[idx[np.abs(t[:, None] - idx[None, :]).argmin(axis=1)]]


def pose_face_reference(pose_xy: np.ndarray, forehead_k: float = FOREHEAD_K, chin_k: float = CHIN_K) -> dict:
    """pose_xy: (T, >=11, >=2) -> {face_mesh_index: (T, 2)} for the four points sign_features_v3 reads.
    Pose indices: 0 nose, 2/5 eye centres, 9/10 mouth corners."""
    nose = pose_xy[:, 0, :2]
    eye = (pose_xy[:, 2, :2] + pose_xy[:, 5, :2]) / 2.0
    mouth = (pose_xy[:, 9, :2] + pose_xy[:, 10, :2]) / 2.0
    axis = eye - mouth
    return {_FACE_NOSE: nose, _FACE_CHIN: mouth - chin_k * axis,
            _FACE_MOUTH: mouth, _FACE_FOREHEAD: eye + forehead_k * axis}


def prepare_landmarks(arr: np.ndarray, aspect: float | None, *, n_frames: int = EXTRACT_FRAMES,
                      forehead_k: float = FOREHEAD_K, chin_k: float = CHIN_K) -> np.ndarray:
    """(T, 543, 3) raw browser landmarks -> (n_frames, 543, 3) laid out as in training."""
    out = resample_landmarks(np.asarray(arr, dtype=np.float32), n_frames)
    out[:, _FACE] = 0.0                                   # the live face mesh is never used
    out[:, _POSE] = _fill_missing_pose(out[:, _POSE])
    if aspect:
        out[:, :, 0] *= float(aspect)                     # zeros (missing landmarks) stay zero
    for idx, pts in pose_face_reference(out[:, _POSE], forehead_k, chin_k).items():
        out[:, 33 + idx, :2] = pts
    return out


# ==========================================================================
# Robust variant of extract_features (sign_features_v4 revision 2; v3 above is unchanged).
# Fixes the two ways v3 mishandles MISSED hands: active-hand choice and mirroring. Identical when no hand is missed.
# ==========================================================================
def _active_secondary_robust(phys, wrists):
    """Which hand is the signing (active) hand. Same rule as v3 (larger wrist travel; ties by presence) but travel is
    measured ONLY between consecutive frames in which the hand was actually seen. v3 measures it on arrays where a
    missed frame is stored as zeros, so every miss adds two fake jumps of about |wrist| and a hand with a few
    dropouts looks like it moves a lot: the wrong hand is chosen and the whole sign may be mirrored."""
    travel = np.zeros(2)
    n_present = np.zeros(2, dtype=int)
    for s in (0, 1):
        present = np.any(wrists[:, s] != 0.0, axis=1)
        n_present[s] = int(present.sum())
        both = present[1:] & present[:-1]
        if both.any():
            step = np.linalg.norm(np.diff(wrists[:, s], axis=0), axis=1)
            travel[s] = float(np.sum(step[both]))
    if travel[0] > travel[1]:
        return 0
    if travel[1] > travel[0]:
        return 1
    return 1 if n_present[1] > n_present[0] else 0


def _mirror_x_present(arr, mid_x):
    """Mirror x about the shoulder centre for landmarks that were DETECTED only. v3 also mirrors the zeros that mean
    'not detected': x = 0 becomes 2 * mid_x, which is non-zero, so every missed hand frame turned into a phantom hand
    at the top of the image after mirroring."""
    out = arr.copy()
    seen = np.any(arr != 0.0, axis=2)                         # (T, K): landmark has any non-zero coordinate
    mirrored = 2.0 * mid_x[:, None] - arr[:, :, 0]
    out[:, :, 0] = np.where(seen, mirrored, 0.0)
    return out


def extract_features_robust(landmarks, layout='signsage', n_frames=N_FRAMES):
    if layout != 'signsage':
        raise ValueError("sign_features_v3 supports layout='signsage' only (543-array)")
    if landmarks.ndim != 3 or landmarks.shape[2] < 2:
        raise ValueError("landmarks must be (T, N, >=3)")
    T = landmarks.shape[0]
    pose = landmarks[:, 0:33, :]
    lh = _xy(landmarks, _LHAND)
    rh = _xy(landmarks, _RHAND)
    face = landmarks[:, 33:500, :2]

    centre, width = _shoulder_centre_width(pose)
    phys, wrists = _track_hands(lh, rh)
    active = _active_secondary_robust(phys, wrists)
    secondary = 1 - active

    mirror = _should_mirror(phys, active, wrists, centre)
    if mirror:
        mid_x = centre[:, 0]
        lh = _mirror_x_present(lh, mid_x)
        rh = _mirror_x_present(rh, mid_x)
        face = _mirror_x_present(face, mid_x)
        phys, wrists = _track_hands(lh, rh)
        active = _active_secondary_robust(phys, wrists)
        secondary = 1 - active

    def hand_part(hfilled, hmask, hsize):
        wrist = hfilled[:, _WRIST, :]
        others = np.delete(hfilled, _WRIST, axis=1)
        wrs = (wrist - centre) / width
        orw = ((others - wrist[:, None, :]) / hsize.reshape(-1, 1, 1))
        wrs = wrs * hmask[:, None]
        orw = orw * hmask[:, None, None]
        return np.concatenate([wrs.reshape(T, -1), orw.reshape(T, -1)], axis=1)

    slot_lm = [np.zeros((T, 21, 2), dtype=np.float64),
               np.zeros((T, 21, 2), dtype=np.float64)]
    for t, phys_t in enumerate(phys):
        for s in (0, 1):
            if phys_t[s] is not None:
                slot_lm[s][t] = phys_t[s]

    l_filled, l_mask = _fill_gaps(slot_lm[0])
    r_filled, r_mask = _fill_gaps(slot_lm[1])

    def hand_size(hfilled):
        return np.linalg.norm(hfilled[:, _WRIST, :] - hfilled[:, _MIDDLE_TIP, :],
                              axis=1, keepdims=True) + EPS

    active_filled = l_filled if active == 0 else r_filled
    active_mask = l_mask if active == 0 else r_mask
    active_size = hand_size(active_filled)
    sec_filled = l_filled if secondary == 0 else r_filled
    sec_mask = l_mask if secondary == 0 else r_mask
    sec_size = hand_size(sec_filled)

    active_hand = hand_part(active_filled, active_mask, active_size)
    sec_hand = hand_part(sec_filled, sec_mask, sec_size)

    h2f = []
    for hp, hmask in ((active_filled[:, _WRIST, :], active_mask),
                      (active_filled[:, _INDEX_TIP, :], active_mask),
                      (sec_filled[:, _WRIST, :], sec_mask),
                      (sec_filled[:, _INDEX_TIP, :], sec_mask)):
        for fidx in _FACE_POINTS:
            h2f.append(((hp - face[:, fidx, :]) / width) * hmask[:, None])
    h2f = np.concatenate(h2f, axis=1)

    def wrist_vel(hfilled, hmask):
        wrist = hfilled[:, _WRIST, :]
        vel = np.zeros_like(wrist)
        if T > 1:
            vel[1:] = wrist[1:] - wrist[:-1]
        return vel * hmask[:, None]

    active_vel = wrist_vel(active_filled, active_mask)
    sec_vel = wrist_vel(sec_filled, sec_mask)

    feat = np.concatenate([active_hand, sec_hand,
                           active_mask.reshape(T, 1), sec_mask.reshape(T, 1),
                           h2f,
                           active_vel, sec_vel], axis=1)
    assert feat.shape[1] == FEAT_DIM, feat.shape[1]
    feat = _resample(feat, n_frames)
    return feat.astype(np.float32)


def extract_features_v4(arr: np.ndarray, aspect: float | None, *, n_frames: int = EXTRACT_FRAMES,
                        forehead_k: float = FOREHEAD_K, chin_k: float = CHIN_K, robust: bool = True) -> np.ndarray:
    """(T, 543, 3) raw landmarks -> (60, 122) float32 features for the FDMSE-ISL models.

    robust=True (default) selects the signing hand from frames where the hand was actually seen; it is identical to
    the legacy sign_features_v3 behaviour when no hand is missed, and far more stable when a webcam misses hands.
    robust=False reproduces the legacy behaviour exactly (used by the golden test of features built without it)."""
    prepared = prepare_landmarks(arr, aspect, n_frames=n_frames, forehead_k=forehead_k, chin_k=chin_k)
    return extract_features_robust(prepared) if robust else _extract_v3(prepared, layout="signsage")


__all__ = ["EXTRACT_FRAMES", "FEAT_DIM", "N_FRAMES", "extract_features_robust", "extract_features_v4",
           "prepare_landmarks", "pose_face_reference", "resample_landmarks"]


In [ ]:
%%writefile sp_aug.py
"""Landmark-level augmentation for the FDMSE-ISL word model (v2).

The live-path stress test showed what hurts a model trained only on clean studio clips: fewer frames per second,
dropped frames, missed hands, a sign cut off or padded with idle frames, and a wrong camera aspect ratio. This module
perturbs the RAW saved landmarks in exactly those ways and then recomputes features with the SERVING recipe
(serving_v4.extract_features_v4), so training sees what serving will feed the model.

Augmentation is deterministic: the same clip and variant number always give the same result (resumable, reproducible).
"""
from __future__ import annotations

import hashlib
import os
import shutil
import tempfile
import time
from pathlib import Path

import numpy as np

_POSE = slice(0, 33)
_LHAND = slice(501, 522)
_RHAND = slice(522, 543)
MIN_FRAMES = 12

# probability that each perturbation is applied to a variant (they combine independently)
AUG_SPEC = {
    "wrong aspect (x stretched, 0.8x to 1.25x)": 0.30,
    "camera roll (rotation, +-6 degrees)": 0.30,
    "detector noise (sigma 0.001 to 0.006)": 0.50,
    "hand dropout bursts (3% to 25% of frames, runs of about 2.5)": 0.50,
    "random frame drops (5% to 20%)": 0.30,
    "lower frame rate (22 to 47 frames)": 0.60,
    "window error (trim up to 20% OR pad up to 25% with idle frames)": 0.50,
}


def _sample_uniform(arr: np.ndarray, n: int) -> np.ndarray:
    return arr[np.linspace(0, len(arr) - 1, n).round().astype(int)]


def _jitter(a: np.ndarray, sigma: float, rng) -> np.ndarray:
    out = a.copy()
    present = out[..., :2] != 0
    out[..., :2] += (rng.normal(0.0, sigma, out[..., :2].shape) * present).astype(np.float32)
    return out


def _burst_dropout(a: np.ndarray, rng) -> np.ndarray:
    out = a.copy()
    for sl in (_LHAND, _RHAND):
        if rng.rand() < 0.7:
            rate = rng.uniform(0.03, 0.25)
            mean_run = 2.5
            p_start = rate / (mean_run * (1.0 - rate))
            state, mask = False, np.zeros(len(out), bool)
            for t in range(len(out)):
                state = (rng.rand() > 1.0 / mean_run) if state else (rng.rand() < p_start)
                mask[t] = state
            out[mask, sl] = 0
    return out


def _drop_frames(a: np.ndarray, frac: float, rng) -> np.ndarray:
    keep = np.sort(rng.choice(len(a), max(MIN_FRAMES, int(round(len(a) * (1 - frac)))), replace=False))
    return a[keep]


def _window(a: np.ndarray, rng) -> np.ndarray:
    n = len(a)
    if rng.rand() < 0.5:                                   # capture started late / ended early
        head, tail = rng.uniform(0, 0.2), rng.uniform(0, 0.2)
        out = a[int(round(n * head)): n - int(round(n * tail))]
        return out if len(out) >= MIN_FRAMES else a
    pre, post = rng.uniform(0, 0.25), rng.uniform(0, 0.25)  # idle frames before / after (segmenter pre/post-roll)
    return np.concatenate([np.repeat(a[:1], int(round(n * pre)), 0), a, np.repeat(a[-1:], int(round(n * post)), 0)])


def _roll(a: np.ndarray, degrees: float, aspect: float) -> np.ndarray:
    """Rotate about the shoulder centre in square (isotropic) units, then convert back to image units."""
    out = a.copy()
    sh = a[:, [11, 12], :2]
    valid = np.all(sh != 0, axis=(1, 2))
    if not valid.any():
        return out
    c = sh[valid].mean(axis=(0, 1)) * np.array([aspect, 1.0])
    th = np.deg2rad(degrees)
    rot = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]], np.float32)
    xy = out[..., :2] * np.array([aspect, 1.0], np.float32)
    present = out[..., :2] != 0
    present = present.any(axis=-1, keepdims=True)
    rotated = (xy - c) @ rot.T + c
    out[..., :2] = np.where(present, rotated / np.array([aspect, 1.0], np.float32), 0.0)
    return out


def augment_raw(arr: np.ndarray, rng, aspect: float):
    """arr: (T, 543, 3) raw landmarks as the browser would send them (zeros = not detected).
    Returns (perturbed array, aspect value the serving code is told)."""
    a = arr.astype(np.float32, copy=True)
    told_aspect = aspect
    if rng.rand() < 0.30:
        told_aspect = aspect * float(np.exp(rng.uniform(np.log(0.8), np.log(1.25))))
    if rng.rand() < 0.30:
        a = _roll(a, rng.uniform(-6.0, 6.0), aspect)
    if rng.rand() < 0.50:
        a = _jitter(a, rng.uniform(0.001, 0.006), rng)
    if rng.rand() < 0.50:
        a = _burst_dropout(a, rng)
    if rng.rand() < 0.30:
        a = _drop_frames(a, rng.uniform(0.05, 0.20), rng)
    if rng.rand() < 0.60:
        a = _sample_uniform(a, int(rng.randint(22, 47)))
    if rng.rand() < 0.50:
        a = _window(a, rng)
    if len(a) < MIN_FRAMES:
        return arr.astype(np.float32, copy=True), aspect
    return a, told_aspect


def variant_rng(key: str, variant: int, seed: int) -> np.random.RandomState:
    h = int(hashlib.sha1(f"{key}:{variant}:{seed}".encode()).hexdigest()[:8], 16)
    return np.random.RandomState(h)


def _copy_atomic(src, dst, retries=5):
    dst = Path(dst)
    tmp = dst.with_name(dst.name + ".tmp")
    last = None
    for a in range(retries):
        try:
            shutil.copyfile(src, tmp)
            os.replace(tmp, dst)
            return
        except Exception as e:                              # Drive hiccups
            last = e
            time.sleep(2 * (a + 1))
    raise RuntimeError(f"could not copy {src} -> {dst}: {last}")


def build_aug_features(chunk_dir, out_dir, aspect, k, seed, serving, load_npz, log=print):
    """For every raw chunk file, write aug_chunk_#####.npz holding K augmented feature variants per clip.
    Resumable (finished chunks are skipped). serving = the module with extract_features_v4;
    load_npz = a retrying npz reader (sp_train._load_npz_retry)."""
    chunk_dir, out_dir = Path(chunk_dir), Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    for p in out_dir.glob("*.tmp"):
        p.unlink(missing_ok=True)
    def ids(d, prefix):
        return {int(p.stem.split("_")[-1]) for p in d.glob(f"{prefix}_*.npz")}
    todo = sorted(ids(chunk_dir, "chunk") - ids(out_dir, "aug_chunk"))
    log(f"augmented feature files existing={len(ids(out_dir, 'aug_chunk'))} | chunks to build={len(todo)} | variants per clip={k}")
    t0 = time.time()
    failed = []
    for n, cid in enumerate(todo, 1):
        try:
            z = load_npz(chunk_dir / f"chunk_{cid:05d}.npz")
        except BaseException as e:                           # unreadable chunk: skip, rerun later
            failed.append(cid); log(f"  chunk {cid:05d} skipped: {str(e)[:120]}"); continue
        keys, errs = z["keys"].astype(str), z["errs"].astype(str)
        nclip, T = len(keys), z["pose"].shape[1]
        X = np.zeros((nclip, k, 60, 122), np.float16)
        ok = np.zeros((nclip, k), bool)
        for i in range(nclip):
            if errs[i]:
                continue
            arr = np.zeros((T, 543, 3), np.float32)
            arr[:, _POSE] = z["pose"][i]
            arr[:, _LHAND] = z["hands"][i, :, 0]
            arr[:, _RHAND] = z["hands"][i, :, 1]
            for v in range(k):
                a, asp = augment_raw(arr, variant_rng(keys[i], v, seed), aspect)
                try:
                    f = serving.extract_features_v4(a, asp, robust=True)
                except Exception:
                    continue
                if np.isfinite(f).all() and np.abs(f).sum() > 0:
                    X[i, v] = f
                    ok[i, v] = True
        fd, tmp = tempfile.mkstemp(suffix=".npz")
        os.close(fd)
        try:
            with open(tmp, "wb") as fh:
                np.savez_compressed(fh, keys=keys, X=X, ok=ok, seed=np.int64(seed), aspect=np.float64(aspect))
            _copy_atomic(tmp, out_dir / f"aug_chunk_{cid:05d}.npz")
        finally:
            if os.path.exists(tmp):
                os.unlink(tmp)
        if n % 10 == 0 or n == len(todo):
            el = (time.time() - t0) / 60
            log(f"  augmented: {n}/{len(todo)} chunks ({el:.1f} min, about {el/n*(len(todo)-n):.0f} min left)")
    if failed:
        log(f"WARNING: {len(failed)} chunk(s) skipped: {failed[:10]}. Rerun the cell to retry them.")
    return len(todo) - len(failed)


In [ ]:
# ============================ 7. BUILD THE AUGMENTED FEATURES (resumable, 30-60 minutes the first time) ============================
# Each clip is perturbed K times the way live capture would (see the table above) and re-run through the SERVING code.
# If Colab disconnects, rerun cells 1-6 and this cell: finished chunks are skipped.
for m in (importlib.invalidate_caches,):
    m()
import sp_features, sp_train, serving_v4, sp_aug
for m in (sp_features, sp_train, serving_v4, sp_aug):
    importlib.reload(m)
sp_aug.build_aug_features(CHUNKS, AUG_DIR, ASPECT, K_VARIANTS, AUG_SEED, serving_v4, sp_train._load_npz_retry)

In [ ]:
# ============================ 8. LOAD CLEAN + AUGMENTED DATA ============================
manifest = pd.read_csv(f"{META}/manifest.csv")
CLASSES = json.load(open(f"{META}/label_map.json", encoding="utf-8"))["classes"]
data = sp_train.load_dataset(FEATS, manifest)                                  # clean robust features, all splits
S = data["stats"]
cls_of, spl_of = manifest["class_id"].to_numpy(), manifest["split"].to_numpy()
key_to_row = {hashlib.sha1(p.encode()).hexdigest(): i for i, p in enumerate(manifest["path"])}
Xa, ya, sa = [], [], []
n_files = 0
for f in sorted(os.listdir(AUG_DIR)):
    if not f.startswith("aug_chunk_") or not f.endswith(".npz"):
        continue
    z = sp_train._load_npz_retry(f"{AUG_DIR}/{f}")
    rows = np.array([key_to_row.get(k, -1) for k in z["keys"].astype(str)])
    sel = z["ok"] & (rows[:, None] >= 0)
    rr = np.repeat(rows[:, None], z["ok"].shape[1], axis=1)[sel]
    Xa.append(z["X"][sel]); ya.append(cls_of[rr].astype(np.int64)); sa.append(spl_of[rr]); n_files += 1
if not Xa:
    raise SystemExit("No augmented features found. Run cell 7.")
Xa, ya, sa = np.concatenate(Xa), np.concatenate(ya), np.concatenate(sa)
is_tr, is_va, is_te = (data["split"] == s for s in ("train", "validation", "test"))
a_tr = sa == "train"
X_tr = np.concatenate([data["X"][is_tr], Xa[a_tr]]); y_tr = np.concatenate([data["y"][is_tr], ya[a_tr]])
X_va, y_va, X_te, y_te = data["X"][is_va], data["y"][is_va], data["X"][is_te], data["y"][is_te]
print(f"\nclean train {is_tr.sum():,} + augmented train {a_tr.sum():,} = {len(X_tr):,} training clips "
      f"({a_tr.sum() / max(1, is_tr.sum() * K_VARIANTS):.1%} of the possible variants were usable)")
print(f"validation {len(X_va):,} and test {len(X_te):,} stay CLEAN (model selection and the one-time test are comparable with baseline_v1)")
if S["converted"] < S["total_in_manifest"]:
    print("WARNING: not all clips are converted yet - the numbers are NOT final.")
mean, std = sp_train.fit_standardizer(X_tr)
np.savez(f"{RUN_DIR}/standardizer.npz", mean=mean, std=std)                  # notebook 03 reads this for this run
ARCH = dict(n_classes=S["n_classes"], proj=HIDDEN, hidden=HIDDEN, layers=LAYERS, dropout=DROPOUT)
HP = dict(epochs=EPOCHS, batch=BATCH, lr=LR, weight_decay=WEIGHT_DECAY, label_smoothing=LABEL_SMOOTHING, patience=PATIENCE,
          noise=NOISE, mask_p=MASK_P, scale_lo=SCALE_LO, seed=SEED, epoch_size=EPOCH_SIZE, device=DEV,
          k_variants=K_VARIANTS, aug_seed=AUG_SEED)
print(f"memory for features: {(data['X'].nbytes + Xa.nbytes)/2**30:.2f} GB (float16)")

In [ ]:
# ============================ 9. TRAIN (resumable - rerun to continue) ============================
model, state = sp_train.train(ARCH, HP, (X_tr, y_tr), RUN_DIR, mean, std, X_val=X_va, y_val=y_va,
                              log_file=f"{RUN_DIR}/train.log")
hist = pd.DataFrame(state["history"])
if len(hist):
    cols = [c for c in ("epoch", "train_loss", "train_acc", "val_top1", "val_top5", "val_loss", "seconds") if c in hist]
    print(hist[cols].tail(10).round(3).to_string(index=False))

In [ ]:
# ============================ 10. TEST ONCE + CONFIDENCE THRESHOLD + SAFETY CHECKS ============================
report_path = f"{RUN_DIR}/test_report.json"
if os.path.exists(report_path) and not FORCE_RETEST:
    REPORT = json.load(open(report_path))
    print("The test split was already evaluated for this run - showing the saved result (not re-testing).")
else:
    best = torch.load(f"{RUN_DIR}/best.pt", map_location="cpu", weights_only=False)
    best_model = sp_train.build_model(ARCH); best_model.load_state_dict(best["model"]); best_model.eval()
    p_va = sp_train.predict_probs(best_model, X_va, mean, std)
    thr_prec, table = sp_train.calibrate_threshold(p_va, y_va, TARGET_PRECISION)
    rej = sp_train.rejection_checks(best_model, mean, std, X_va, thr_prec)
    probe_max = max(v["max_confidence"] for v in rej.values())
    thr = None if thr_prec is None else round(max(thr_prec, probe_max + 0.01), 2)
    p_te = sp_train.predict_probs(best_model, X_te, mean, std)
    m_te = sp_train.metrics_from_probs(p_te, y_te, ARCH["n_classes"]); m_va = sp_train.metrics_from_probs(p_va, y_va, ARCH["n_classes"])
    sp_train.per_class_report(p_te, y_te, CLASSES).to_csv(f"{RUN_DIR}/test_per_class.csv", index=False)
    table.to_csv(f"{RUN_DIR}/calibration_validation.csv", index=False)
    acc_te = p_te.max(1) >= thr if thr is not None else np.zeros(len(y_te), bool)
    at_thr = dict(coverage=float(acc_te.mean()), precision=float((p_te.argmax(1)[acc_te] == y_te[acc_te]).mean()) if acc_te.any() else None)
    REPORT = dict(run=RUN_NAME, best_epoch=best["epoch"], validation=m_va, test=m_te, confidence_threshold=thr,
                  precision_threshold=thr_prec, probe_max_confidence=probe_max, test_at_threshold=at_thr,
                  rejection_checks=rej, data=S, hyperparameters=HP, architecture=ARCH,
                  augmentation=dict(k_variants=K_VARIANTS, seed=AUG_SEED, spec=sp_aug.AUG_SPEC))
    json.dump(REPORT, open(report_path, "w"), indent=2)
T = REPORT["test"]
print(f"TEST (official signer-independent split, used once; CLEAN clips): top-1 {T['top1']:.3f} | top-5 {T['top5']:.3f} | macro-recall {T['macro_recall']:.3f}")
print(f"best epoch {REPORT['best_epoch']} | validation top-1 {REPORT['validation']['top1']:.3f}")
print("for comparison, baseline_v1 on the same split: top-1 0.821 | top-5 0.951")
print(f"final confidence threshold: {REPORT['confidence_threshold']}  (precision threshold {REPORT['precision_threshold']}, non-sign probes reach {REPORT['probe_max_confidence']:.3f})")
print("on the test split at the final threshold:", REPORT["test_at_threshold"])
print("\nIMPORTANT: this clean number alone does not show the live-condition gain. Run notebook 03 on this run next.")

In [ ]:
# ============================ 11. FINAL RETRAIN ON ALL SIGNERS (for deployment) ============================
# Same recipe, fixed number of epochs (= the best epoch above); clean + augmented clips of train, validation AND test.
FINAL_DIR = f"{RUN_DIR}/final"
if RUN_FINAL_RETRAIN:
    X_all = np.concatenate([data["X"], Xa]); y_all = np.concatenate([data["y"], ya])
    mean_all, std_all = sp_train.fit_standardizer(X_all)
    np.savez(f"{FINAL_DIR}_standardizer.npz", mean=mean_all, std=std_all)
    final_epochs = max(5, int(REPORT["best_epoch"]))
    final_model, fstate = sp_train.train(ARCH, HP, (X_all, y_all), FINAL_DIR, mean_all, std_all,
                                         fixed_epochs=final_epochs, log_file=f"{FINAL_DIR}/train.log")
else:
    print("skipped (RUN_FINAL_RETRAIN = False)")

In [ ]:
# ============================ 12. EXPORT + MOTION FLOOR ============================
ext_cfg = json.load(open(f"{META}/extraction_config.json"))
if RUN_FINAL_RETRAIN:
    exp_model, exp_mean, exp_std = final_model, mean_all, std_all
    note = (f"Deployment model: trained on all signers for {final_epochs} epochs on clean + landmark-augmented clips "
            "(frame rate, dropped frames, missed hands, window errors, aspect, noise, roll). Metrics come from the first run "
            "(official split, test used once, clean clips).")
else:
    exp_model = sp_train.build_model(ARCH)
    exp_model.load_state_dict(torch.load(f"{RUN_DIR}/best.pt", map_location="cpu", weights_only=False)["model"])
    exp_mean, exp_std = mean, std
    note = "Trained on the official training split only (clean + augmented); best epoch chosen on validation."
EXPORT_DIR = f"{RUN_DIR}/export"
sp_train.export_bundle(EXPORT_DIR, exp_model, ARCH, exp_mean, exp_std, CLASSES, ext_cfg,
                       dict(test=REPORT["test"], validation=REPORT["validation"], data=REPORT["data"],
                            test_at_threshold=REPORT["test_at_threshold"], rejection_checks=REPORT["rejection_checks"],
                            augmentation=REPORT["augmentation"]),
                       REPORT["confidence_threshold"], note, feature_spec=FEATURE_SPEC)
mf = np.abs(data["X"][:, :, 118:122].astype(np.float32)).mean(axis=(1, 2))      # motion floor from the CLEAN real clips
pcts = {str(p): float(np.percentile(mf, p)) for p in (0.1, 1, 5, 25, 50, 75, 95)}
mj = json.load(open(f"{EXPORT_DIR}/metrics.json"))
mj["motion"] = {"min_motion_feature": 0.5 * pcts["1"], "rule": "half of the 1st percentile over all clean real clips", "percentiles": pcts}
json.dump(mj, open(f"{EXPORT_DIR}/metrics.json", "w"), indent=2)
b = sp_train.load_bundle(EXPORT_DIR)                                              # reload check
sp_train.predict_probs(b["model"], X_va[:4], b["mean"], b["std"])
print("export reloads OK ->", EXPORT_DIR, "| files:", sorted(os.listdir(EXPORT_DIR)))
print("motion floor:", round(mj["motion"]["min_motion_feature"], 5))

## What next
1. **Run notebook 03 on this run**: `RUN_NAME = "aug_v1"`, `FEATURE_SET = "v4r2_pose_face"`, `COMPARE_LEGACY = False`. Compare its table with the one from `baseline_v1`. The rows that matter are G, H, B-D, I-K, L and N.
2. Use `training/aug_v1/export/` in the backend only if notebook 03 shows the live-condition rows clearly improved and the clean test number did not drop noticeably.
3. The model stays **not promoted** until a real webcam has been tried and the gates in `docs/MODELS_AND_DATA.md` section 6 pass.